# B1.15 · Securing the developers' coding agents

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *Security of AI*

Builds on **[B1.14 · Injection in your own pipeline](https://spbreed.github.io/cyber-commons/lessons/B1.14.html)**.

| | |
|---|---|
| Open-source tooling | Docker, Cilium |
| Open-weight models | GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


The coding agent in a developer's IDE is the most privileged agent in most
organisations and the least governed. It sits upstream of everything this track
has built: it writes the code the pipeline later analyses.

What it holds by default:

- the developer's **git credentials** — push access to everything they can push to,
- the **whole monorepo** on disk, including files they never open,
- a **shell**, usually unrestricted,
- their **cloud credentials** in `~/.aws` or `~/.config/gcloud`,
- whatever **MCP servers** they connected, each with its own authority.

That is a production identity in an unmanaged environment, driven by a model
reading code from the internet.

The binding constraint here is not technical feasibility — it is **developer
tolerance**. A containment scheme that adds friction to the inner loop is
disabled within a week, and a disabled control protects nothing. So the design
goal is the strongest containment a developer does not notice.

## 2 · Demo — measure the default configuration

In [ ]:
SCOPE_WEIGHT = {"self":1,"project":3,"tenant":8,"org":20}
DEV_TOOLS = [
 # (name, writes, scope, reversible, needed in the inner loop)
 ("read_file",  False,"self",   True,  True),
 ("write_file", True, "project",True,  True),
 ("run_tests",  True, "self",   True,  True),
 ("run_shell",  True, "tenant", False, True),
 ("git_commit", True, "project",True,  True),
 ("git_push",   True, "project",False, False),
 ("read_env",   False,"org",    True,  False),
 ("http_get",   False,"self",   True,  True),
]
def blast(tools, gated=frozenset()):
    return sum(SCOPE_WEIGHT[s]*(1 if rev else 2)
               for n,w,s,rev,_ in tools if w and n not in gated)

print(f"{'tool':14s}{'writes':8s}{'scope':9s}{'reversible':12s}inner loop?")
print("-" * 58)
for n,w,s,rev,need in DEV_TOOLS:
    print(f"{n:14s}{str(w):8s}{s:9s}{str(rev):12s}{need}")
print(f"\ndefault blast radius: {blast(DEV_TOOLS)}")

In [ ]:
import fnmatch
HOME = [
 "/home/dana/work/monorepo/src/app.py",
 "/home/dana/work/monorepo/.env",
 "/home/dana/work/other-team-repo/secrets.yaml",
 "/home/dana/.aws/credentials",
 "/home/dana/.ssh/id_ed25519",
 "/home/dana/.config/gcloud/application_default_credentials.json",
 "/home/dana/Downloads/customer-export-2026.csv",
]
def normalise(p):
    parts = []
    for seg in p.split("/"):
        if seg in ("", "."): continue
        if seg == "..":
            if parts: parts.pop()
            continue
        parts.append(seg)
    return "/" + "/".join(parts)

DENY = ("*/.ssh/*","*/.aws/*","*/.config/gcloud/*","*.pem","*/.env","*/Downloads/*")
def contained(p, workspace="/home/dana/work/monorepo"):
    real = normalise(p)
    if any(fnmatch.fnmatch(real, g) for g in DENY): return False
    return real.startswith(workspace + "/")

print(f"{'path':64s}{'default':9s}contained")
print("-" * 84)
for p in HOME:
    print(f"{p:64s}{'True':9s}{contained(p)}")
creds = [p for p in HOME if any(k in p for k in (".aws",".ssh","gcloud",".env"))]
print(f"\ncredential files reachable by default: {len(creds)}")
print(f"credential files reachable when contained: "
      f"{sum(1 for p in creds if contained(p))}")

## 3 · The control — rank by friction, ship the invisible ones first

In [ ]:
CONTROLS = [
 ("deny-list credential paths", 0.0,
  "agent cannot read ~/.aws, ~/.ssh, ~/.env. Developers never noticed."),
 ("workspace confinement",      0.1,
  "agent sees the open repo only. Occasionally annoying for monorepo hops."),
 ("egress allowlist",           0.2,
  "package registries + your VCS. Breaks the odd curl in a generated script."),
 ("gate git_push",              0.4,
  "one confirmation before code leaves the machine. Noticed, usually accepted."),
 ("gate every write",           0.9,
  "confirmation per file write. Abandoned within a week, every time."),
 ("no shell at all",            1.0,
  "removes the inner loop. Nobody will use the agent."),
]
print(f"{'control':30s}{'friction':>9}  effect")
print("-" * 96)
for name, fr, note in CONTROLS:
    print(f"{name:30s}{fr:>9.1f}  {'█'*int(fr*10):10s} {note}")
shippable = [c[0] for c in CONTROLS if c[1] <= 0.4]
print(f"\nship now (friction ≤ 0.4): {shippable}")

In [ ]:
gated = {"git_push"}
print(f"blast radius     {blast(DEV_TOOLS):>3} → {blast(DEV_TOOLS, gated):>3}")
print(f"reachable files  {len(HOME):>3} → {sum(1 for p in HOME if contained(p)):>3}")
print(f"credentials      {len(creds):>3} → {sum(1 for p in creds if contained(p)):>3}")
print(f"friction added   0.4 of 1.0 — one confirmation before a push")
assert not any(contained(p) for p in creds)
print("\nNo cloud or SSH credential is reachable, the inner loop is unchanged,")
print("and the only thing a developer notices is a prompt before pushing.")

## What you just proved

The default developer agent scores a blast radius of 43 and can reach all seven paths including AWS, SSH and gcloud credentials. Containment reduces reachable paths to one source file with zero credentials reachable, and gating `git_push` drops the blast radius to 37 for 0.4 friction. The three lowest-friction controls remove every credential path without touching the inner loop.

## Your turn

Ship the credential deny-list first — it is a config file, it takes an afternoon, and no developer will notice. Then find out how many agents in your organisation could read `~/.aws/credentials` yesterday.

---

**Next → [B1.16 · Bonus — Google Mantis, the pipeline in production](https://spbreed.github.io/cyber-commons/lessons/B1.16.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.15.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.15.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*